<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/03_Missingness_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# 03.1 LOAD PROCESSED TRAINING DATA
# ============================================================

from google.colab import drive

try:
    drive.mount(
        "/content/drive",
        force_remount=False
    )
except ValueError:
    print("Google Drive is already mounted.")

from pathlib import Path
import numpy as np
import pandas as pd
import json
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

SPLIT_ROOT = (
    PROJECT_ROOT /
    "data" /
    "splits"
)

SCENARIO_ROOT = (
    PROJECT_ROOT /
    "experiments" /
    "missingness"
)

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

RANDOM_SEEDS = [
    42,
    123,
    2024
]

MISSINGNESS_LEVELS = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]

for dataset_id in DATASET_IDS:

    dataset_dir = (
        SPLIT_ROOT /
        dataset_id
    )

    train_path = (
        dataset_dir /
        "X_train.csv"
    )

    if not train_path.exists():

        raise FileNotFoundError(
            f"Training data not found:\n"
            f"{train_path}\n\n"
            f"Run Notebook 02 first."
        )

TRAINING_DATA = {}

for dataset_id in DATASET_IDS:

    train_path = (
        SPLIT_ROOT /
        dataset_id /
        "X_train.csv"
    )

    TRAINING_DATA[
        dataset_id
    ] = pd.read_csv(
        train_path,
        low_memory=False
    )

    print(
        f"{dataset_id:20s}"
        f" {TRAINING_DATA[dataset_id].shape}"
    )

print("=" * 90)
print("TRAINING DATA LOADED")
print("=" * 90)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
adult_income         (19522, 14)
bank_marketing       (27126, 16)
diabetes_130us       (61059, 47)
TRAINING DATA LOADED


In [3]:
# ============================================================
# 03.2 DEFINE MISSINGNESS MASKS
# ============================================================

FEATURE_REGISTRY = {}

for dataset_id, df in TRAINING_DATA.items():

    numerical = [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(
            df[c]
        )
    ]

    categorical = [
        c for c in df.columns
        if c not in numerical
    ]

    FEATURE_REGISTRY[
        dataset_id
    ] = {
        "numerical": numerical,
        "categorical": categorical,
        "all_features": list(df.columns)
    }

    print("=" * 90)
    print(dataset_id)
    print(
        f"Numerical   : {len(numerical)}"
    )
    print(
        f"Categorical : {len(categorical)}"
    )

adult_income
Numerical   : 6
Categorical : 8
bank_marketing
Numerical   : 7
Categorical : 9
diabetes_130us
Numerical   : 11
Categorical : 36


In [4]:
# ============================================================
# 03.3 MCAR GENERATION
# ============================================================

def generate_mcar_mask(
    df,
    missing_rate,
    seed,
    eligible_columns=None
):

    rng = np.random.default_rng(seed)

    if eligible_columns is None:
        eligible_columns = list(df.columns)

    mask = pd.DataFrame(
        False,
        index=df.index,
        columns=df.columns
    )

    eligible = [
        c for c in eligible_columns
        if c in df.columns
    ]

    total_cells = (
        len(df) *
        len(eligible)
    )

    target_cells = int(
        round(
            total_cells *
            missing_rate
        )
    )

    if target_cells == 0:
        return mask

    selected = rng.choice(
        total_cells,
        size=min(
            target_cells,
            total_cells
        ),
        replace=False
    )

    for position in selected:

        row_idx = (
            position //
            len(eligible)
        )

        col_idx = (
            position %
            len(eligible)
        )

        mask.iloc[
            row_idx,
            df.columns.get_loc(
                eligible[col_idx]
            )
        ] = True

    return mask

In [5]:
# ============================================================
# 03.4 MAR GENERATION
# ============================================================

def generate_mar_mask(
    df,
    missing_rate,
    seed,
    eligible_columns=None
):

    rng = np.random.default_rng(seed)

    if eligible_columns is None:
        eligible_columns = list(df.columns)

    eligible_columns = [
        c for c in eligible_columns
        if c in df.columns
    ]

    mask = pd.DataFrame(
        False,
        index=df.index,
        columns=df.columns
    )

    if not eligible_columns:
        return mask

    # Use observed feature information to
    # construct heterogeneous missingness
    # probabilities.

    scores = np.zeros(len(df))

    numeric_columns = [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(
            df[c]
        )
    ]

    if numeric_columns:

        reference = numeric_columns[0]

        values = (
            df[reference]
            .fillna(
                df[reference].median()
            )
        )

        std = values.std()

        if std > 0:

            scores = (
                values - values.mean()
            ) / std

    probabilities = (
        1 /
        (
            1 +
            np.exp(
                -scores
            )
        )
    )

    if probabilities.max() > probabilities.min():

        probabilities = (
            probabilities -
            probabilities.min()
        ) / (
            probabilities.max() -
            probabilities.min()
        )

    probabilities = (
        0.25 +
        0.75 * probabilities
    )

    for column in eligible_columns:

        uniform = rng.random(
            len(df)
        )

        column_mask = (
            uniform <
            (
                probabilities *
                missing_rate /
                probabilities.mean()
            )
        )

        mask.loc[
            column_mask,
            column
        ] = True

    return mask

In [6]:
# ============================================================
# 03.5 MNAR GENERATION
# ============================================================

def generate_mnar_mask(
    df,
    missing_rate,
    seed,
    eligible_columns=None
):

    rng = np.random.default_rng(seed)

    if eligible_columns is None:
        eligible_columns = list(df.columns)

    eligible_columns = [
        c for c in eligible_columns
        if c in df.columns
    ]

    mask = pd.DataFrame(
        False,
        index=df.index,
        columns=df.columns
    )

    for column in eligible_columns:

        series = df[column]

        if pd.api.types.is_numeric_dtype(
            series
        ):

            values = series.copy()

            threshold = values.quantile(
                0.75
            )

            high_value = (
                values >= threshold
            ).fillna(False)

            probabilities = np.where(
                high_value,
                missing_rate * 1.5,
                missing_rate * 0.5
            )

        else:

            frequencies = (
                series
                .value_counts(
                    normalize=True
                )
            )

            rarity = (
                series.map(
                    frequencies
                )
                .fillna(0)
            )

            probabilities = (
                missing_rate *
                (
                    1.5 -
                    rarity
                )
            )

        probabilities = np.clip(
            probabilities,
            0,
            1
        )

        random_values = rng.random(
            len(df)
        )

        column_mask = (
            random_values <
            probabilities
        )

        mask.loc[
            column_mask,
            column
        ] = True

    return mask

In [7]:
# ============================================================
# 03.6 10% MISSINGNESS
# ============================================================

RATE_10 = 0.10

print(
    f"10% missingness scenario configured: "
    f"{RATE_10:.0%}"
)

10% missingness scenario configured: 10%


In [8]:
# ============================================================
# 03.7 20% MISSINGNESS
# ============================================================

RATE_20 = 0.20

print(
    f"20% missingness scenario configured: "
    f"{RATE_20:.0%}"
)

20% missingness scenario configured: 20%


In [9]:
# ============================================================
# 03.8 30% MISSINGNESS
# ============================================================

RATE_30 = 0.30

print(
    f"30% missingness scenario configured: "
    f"{RATE_30:.0%}"
)

30% missingness scenario configured: 30%


In [10]:
# ============================================================
# 03.9 40% MISSINGNESS
# ============================================================

RATE_40 = 0.40

print(
    f"40% missingness scenario configured: "
    f"{RATE_40:.0%}"
)

40% missingness scenario configured: 40%


In [11]:
# ============================================================
# 03.10 50% MISSINGNESS
# ============================================================

RATE_50 = 0.50

print(
    f"50% missingness scenario configured: "
    f"{RATE_50:.0%}"
)

50% missingness scenario configured: 50%


In [12]:
# ============================================================
# 03.6–03.10 MISSINGNESS LEVEL GENERATION
# ============================================================

def apply_missingness(
    df,
    mask
):

    result = df.copy()

    result = result.astype(
        object
    )

    result[mask] = np.nan

    return result


def generate_scenario(
    df,
    mechanism,
    missing_rate,
    seed
):

    if mechanism == "MCAR":

        mask = generate_mcar_mask(
            df,
            missing_rate,
            seed
        )

    elif mechanism == "MAR":

        mask = generate_mar_mask(
            df,
            missing_rate,
            seed
        )

    elif mechanism == "MNAR":

        mask = generate_mnar_mask(
            df,
            missing_rate,
            seed
        )

    else:

        raise ValueError(
            f"Unknown mechanism: "
            f"{mechanism}"
        )

    masked_df = apply_missingness(
        df,
        mask
    )

    return masked_df, mask


MECHANISMS = [
    "MCAR",
    "MAR",
    "MNAR"
]

print(
    "Missingness levels:",
    MISSINGNESS_LEVELS
)

Missingness levels: [0.1, 0.2, 0.3, 0.4, 0.5]


In [13]:
# ============================================================
# 03.11 CELL-WISE MASKING — OPTIMIZED
# ============================================================

import gc
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

SCENARIO_ROOT = (
    PROJECT_ROOT /
    "experiments" /
    "missingness"
)

TARGET_REGISTRY = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted"
}

MECHANISMS = [
    "MCAR",
    "MAR",
    "MNAR"
]

MISSINGNESS_LEVELS = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]

RANDOM_SEEDS = [
    42,
    123,
    2024
]

SCENARIO_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# FAST CELL-WISE MASK GENERATOR
# ------------------------------------------------------------

def generate_fast_cell_mask(
    df,
    mechanism,
    missing_rate,
    seed,
    target_column=None
):

    rng = np.random.default_rng(seed)

    eligible_columns = [
        column
        for column in df.columns
        if column != target_column
    ]

    n_rows = len(df)
    n_columns = len(eligible_columns)

    mask = np.zeros(
        (n_rows, n_columns),
        dtype=bool
    )

    if n_columns == 0:
        return mask, eligible_columns

    # ========================================================
    # MCAR
    # ========================================================

    if mechanism == "MCAR":

        total_cells = (
            n_rows *
            n_columns
        )

        n_missing = int(
            round(
                total_cells *
                missing_rate
            )
        )

        if n_missing > 0:

            selected = rng.choice(
                total_cells,
                size=min(
                    n_missing,
                    total_cells
                ),
                replace=False
            )

            row_indices = (
                selected //
                n_columns
            )

            column_indices = (
                selected %
                n_columns
            )

            mask[
                row_indices,
                column_indices
            ] = True

    # ========================================================
    # MAR
    # ========================================================

    elif mechanism == "MAR":

        numerical_columns = [
            column
            for column in eligible_columns
            if pd.api.types.is_numeric_dtype(
                df[column]
            )
        ]

        if numerical_columns:

            reference_column = (
                numerical_columns[0]
            )

            reference = pd.to_numeric(
                df[reference_column],
                errors="coerce"
            )

            reference = reference.fillna(
                reference.median()
            )

            median_value = reference.median()

            high_group = (
                reference >= median_value
            ).to_numpy()

            high_probability = min(
                missing_rate * 1.5,
                1.0
            )

            low_probability = min(
                missing_rate * 0.5,
                1.0
            )

            probabilities = np.where(
                high_group,
                high_probability,
                low_probability
            )

            random_matrix = rng.random(
                (
                    n_rows,
                    n_columns
                )
            )

            mask = (
                random_matrix <
                probabilities[:, None]
            )

        else:

            random_matrix = rng.random(
                (
                    n_rows,
                    n_columns
                )
            )

            mask = (
                random_matrix <
                missing_rate
            )

    # ========================================================
    # MNAR
    # ========================================================

    elif mechanism == "MNAR":

        for column_index, column in enumerate(
            eligible_columns
        ):

            series = df[column]

            if pd.api.types.is_numeric_dtype(
                series
            ):

                values = pd.to_numeric(
                    series,
                    errors="coerce"
                )

                threshold = values.quantile(
                    0.75
                )

                high_value = (
                    values >= threshold
                ).fillna(False).to_numpy()

                probabilities = np.where(
                    high_value,
                    missing_rate * 1.5,
                    missing_rate * 0.5
                )

            else:

                frequencies = (
                    series
                    .value_counts(
                        normalize=True
                    )
                )

                rarity = (
                    series
                    .map(frequencies)
                    .fillna(0)
                    .to_numpy()
                )

                probabilities = (
                    missing_rate *
                    np.clip(
                        1.5 - rarity,
                        0.5,
                        1.5
                    )
                )

            probabilities = np.clip(
                probabilities,
                0.0,
                1.0
            )

            random_values = rng.random(
                n_rows
            )

            mask[:, column_index] = (
                random_values <
                probabilities
            )

    else:

        raise ValueError(
            f"Unsupported mechanism: "
            f"{mechanism}"
        )

    return mask, eligible_columns


print(
    "Optimized cell-wise masking function ready."
)

Optimized cell-wise masking function ready.


In [14]:
# ============================================================
# 03.11B GENERATE AND SAVE CELL-WISE SCENARIOS
# ============================================================

SCENARIO_REGISTRY = []

for dataset_id in DATASET_IDS:

    df = TRAINING_DATA[dataset_id]

    target_column = TARGET_REGISTRY.get(
        dataset_id
    )

    dataset_dir = (
        SCENARIO_ROOT /
        dataset_id
    )

    dataset_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    print()
    print("=" * 90)
    print(
        f"DATASET: {dataset_id}"
    )
    print("=" * 90)

    for mechanism in MECHANISMS:

        for missing_rate in MISSINGNESS_LEVELS:

            for seed in RANDOM_SEEDS:

                scenario_name = (
                    f"{mechanism.lower()}_"
                    f"{int(missing_rate * 100)}pct_"
                    f"seed_{seed}"
                )

                data_path = (
                    dataset_dir /
                    f"{scenario_name}.csv"
                )

                mask_path = (
                    dataset_dir /
                    f"{scenario_name}_mask.csv"
                )

                # ------------------------------------------------
                # Skip if already generated
                # ------------------------------------------------

                if (
                    data_path.exists()
                    and
                    mask_path.exists()
                ):

                    print(
                        f"SKIP  {scenario_name}"
                    )

                    SCENARIO_REGISTRY.append({

                        "dataset_id":
                            dataset_id,

                        "scenario":
                            scenario_name,

                        "mechanism":
                            mechanism,

                        "missing_rate":
                            missing_rate,

                        "seed":
                            seed,

                        "data_path":
                            str(data_path),

                        "mask_path":
                            str(mask_path)
                    })

                    continue

                # ------------------------------------------------
                # Generate mask
                # ------------------------------------------------

                mask_array, eligible_columns = (
                    generate_fast_cell_mask(
                        df=df,
                        mechanism=mechanism,
                        missing_rate=missing_rate,
                        seed=seed,
                        target_column=target_column
                    )
                )

                # ------------------------------------------------
                # Create masked dataset
                # ------------------------------------------------

                masked_df = df.copy()

                masked_values = (
                    masked_df[
                        eligible_columns
                    ].to_numpy(
                        dtype=object
                    )
                )

                masked_values[
                    mask_array
                ] = np.nan

                masked_df[
                    eligible_columns
                ] = masked_values

                # ------------------------------------------------
                # Protect target
                # ------------------------------------------------

                if target_column in df.columns:

                    masked_df[
                        target_column
                    ] = df[
                        target_column
                    ]

                # ------------------------------------------------
                # Save masked dataset
                # ------------------------------------------------

                masked_df.to_csv(
                    data_path,
                    index=False
                )

                # ------------------------------------------------
                # Save mask
                # ------------------------------------------------

                mask_df = pd.DataFrame(
                    mask_array.astype(
                        np.uint8
                    ),
                    columns=eligible_columns
                )

                mask_df.to_csv(
                    mask_path,
                    index=False
                )

                actual_rate = float(
                    mask_array.mean()
                )

                SCENARIO_REGISTRY.append({

                    "dataset_id":
                        dataset_id,

                    "scenario":
                        scenario_name,

                    "mechanism":
                        mechanism,

                    "missing_rate":
                        float(
                            missing_rate
                        ),

                    "actual_missing_rate":
                        actual_rate,

                    "seed":
                        int(seed),

                    "target_column":
                        target_column,

                    "rows":
                        int(len(df)),

                    "features":
                        int(
                            len(
                                eligible_columns
                            )
                        ),

                    "missing_cells":
                        int(
                            mask_array.sum()
                        ),

                    "data_path":
                        str(data_path),

                    "mask_path":
                        str(mask_path)
                })

                print(
                    f"SAVED {scenario_name} "
                    f"| actual={actual_rate:.4f}"
                )

                # ------------------------------------------------
                # Release memory
                # ------------------------------------------------

                del masked_df
                del masked_values
                del mask_df
                del mask_array

                gc.collect()

print()
print("=" * 90)
print(
    "CELL-WISE SCENARIO GENERATION COMPLETED"
)
print("=" * 90)


DATASET: adult_income
SKIP  mcar_10pct_seed_42
SKIP  mcar_10pct_seed_123
SKIP  mcar_10pct_seed_2024
SKIP  mcar_20pct_seed_42
SKIP  mcar_20pct_seed_123
SKIP  mcar_20pct_seed_2024
SKIP  mcar_30pct_seed_42
SKIP  mcar_30pct_seed_123
SKIP  mcar_30pct_seed_2024
SKIP  mcar_40pct_seed_42
SKIP  mcar_40pct_seed_123
SKIP  mcar_40pct_seed_2024
SKIP  mcar_50pct_seed_42
SKIP  mcar_50pct_seed_123
SKIP  mcar_50pct_seed_2024
SKIP  mar_10pct_seed_42
SKIP  mar_10pct_seed_123
SKIP  mar_10pct_seed_2024
SKIP  mar_20pct_seed_42
SKIP  mar_20pct_seed_123
SKIP  mar_20pct_seed_2024
SKIP  mar_30pct_seed_42
SKIP  mar_30pct_seed_123
SKIP  mar_30pct_seed_2024
SKIP  mar_40pct_seed_42
SKIP  mar_40pct_seed_123
SKIP  mar_40pct_seed_2024
SKIP  mar_50pct_seed_42
SKIP  mar_50pct_seed_123
SKIP  mar_50pct_seed_2024
SKIP  mnar_10pct_seed_42
SKIP  mnar_10pct_seed_123
SKIP  mnar_10pct_seed_2024
SKIP  mnar_20pct_seed_42
SKIP  mnar_20pct_seed_123
SKIP  mnar_20pct_seed_2024
SKIP  mnar_30pct_seed_42
SKIP  mnar_30pct_seed_123
SKIP 

In [15]:
# ============================================================
# 03.11C SAVE CELL-WISE SCENARIO REGISTRY
# ============================================================

CELL_WISE_REGISTRY_DF = pd.DataFrame(
    SCENARIO_REGISTRY
)

REGISTRY_PATH = (
    SCENARIO_ROOT /
    "cell_wise_scenario_registry.csv"
)

CELL_WISE_REGISTRY_DF.to_csv(
    REGISTRY_PATH,
    index=False
)

print("=" * 90)
print("CELL-WISE SCENARIO REGISTRY")
print("=" * 90)

print(
    f"Total scenarios: "
    f"{len(CELL_WISE_REGISTRY_DF):,}"
)

print(
    f"Saved to:\n"
    f"{REGISTRY_PATH}"
)

display(
    CELL_WISE_REGISTRY_DF.head()
)

CELL-WISE SCENARIO REGISTRY
Total scenarios: 135
Saved to:
/content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/cell_wise_scenario_registry.csv


,dataset_id,scenario,mechanism,missing_rate,seed,data_path,mask_path
0,adult_income,mcar_10pct_seed_42,MCAR,0.1,42,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
1,adult_income,mcar_10pct_seed_123,MCAR,0.1,123,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
2,adult_income,mcar_10pct_seed_2024,MCAR,0.1,2024,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
3,adult_income,mcar_20pct_seed_42,MCAR,0.2,42,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
4,adult_income,mcar_20pct_seed_123,MCAR,0.2,123,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...


In [16]:
# ============================================================
# 03.12 FEATURE-WISE MASKING — MEMORY-EFFICIENT
# ============================================================

import gc
import numpy as np
import pandas as pd

FEATURE_SCENARIO_ROOT = (
    SCENARIO_ROOT /
    "feature_wise"
)

FEATURE_SCENARIO_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 90)
print("FEATURE-WISE MASKING CONFIGURED")
print("=" * 90)

FEATURE-WISE MASKING CONFIGURED


In [17]:
# ============================================================
# 03.13 MULTIPLE RANDOM SEEDS
# ============================================================

import json
from pathlib import Path

# ------------------------------------------------------------
# RESEARCH REPRODUCIBILITY SEED CONFIGURATION
# ------------------------------------------------------------

SEED_REGISTRY = {

    "primary_seed": 42,

    "replication_seeds": [
        123,
        2024
    ],

    "all_seeds": [
        42,
        123,
        2024
    ],

    "number_of_seeds": 3,

    "purpose": (
        "Primary experiment plus two independent "
        "replication seeds for robustness analysis."
    )
}

# ------------------------------------------------------------
# VERIFY CONSISTENCY WITH NOTEBOOK 03
# ------------------------------------------------------------

EXPECTED_SEEDS = [
    42,
    123,
    2024
]

if RANDOM_SEEDS != EXPECTED_SEEDS:

    raise ValueError(
        "RANDOM_SEEDS does not match "
        "the research seed registry.\n"
        f"RANDOM_SEEDS = {RANDOM_SEEDS}\n"
        f"Expected      = {EXPECTED_SEEDS}"
    )

# ------------------------------------------------------------
# VERIFY UNIQUENESS
# ------------------------------------------------------------

if len(
    SEED_REGISTRY["all_seeds"]
) != len(
    set(
        SEED_REGISTRY["all_seeds"]
    )
):

    raise ValueError(
        "Duplicate random seeds detected."
    )

# ------------------------------------------------------------
# SAVE SEED REGISTRY
# ------------------------------------------------------------

SEED_REGISTRY_PATH = (
    SCENARIO_ROOT /
    "seed_registry.json"
)

with open(
    SEED_REGISTRY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        SEED_REGISTRY,
        f,
        indent=4
    )

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("=" * 90)
print("03.13 MULTIPLE RANDOM SEEDS")
print("=" * 90)

print(
    json.dumps(
        SEED_REGISTRY,
        indent=4
    )
)

print()
print(
    f"Seed registry saved to:\n"
    f"{SEED_REGISTRY_PATH}"
)

print()
print(
    "Seed configuration verified successfully."
)

03.13 MULTIPLE RANDOM SEEDS
{
    "primary_seed": 42,
    "replication_seeds": [
        123,
        2024
    ],
    "all_seeds": [
        42,
        123,
        2024
    ],
    "number_of_seeds": 3,
    "purpose": "Primary experiment plus two independent replication seeds for robustness analysis."
}

Seed registry saved to:
/content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/seed_registry.json

Seed configuration verified successfully.


In [18]:
# ============================================================
# 03.14 GROUND-TRUTH PRESERVATION
# ============================================================

import hashlib
import pandas as pd

GROUND_TRUTH_REGISTRY = []

GROUND_TRUTH_ROOT = (
    PROJECT_ROOT /
    "data" /
    "splits"
)

for dataset_id in DATASET_IDS:

    ground_truth_path = (
        GROUND_TRUTH_ROOT /
        dataset_id /
        "X_train.csv"
    )

    if not ground_truth_path.exists():

        raise FileNotFoundError(
            f"Ground-truth training data not found:\n"
            f"{ground_truth_path}"
        )

    # --------------------------------------------------------
    # Read only for verification.
    # No permanent copy is retained.
    # --------------------------------------------------------

    ground_truth_df = pd.read_csv(
        ground_truth_path,
        low_memory=False
    )

    # --------------------------------------------------------
    # Verify against currently loaded training data
    # --------------------------------------------------------

    current_df = TRAINING_DATA[
        dataset_id
    ]

    if not ground_truth_df.equals(
        current_df
    ):

        raise AssertionError(
            f"Ground-truth mismatch detected "
            f"for {dataset_id}."
        )

    # --------------------------------------------------------
    # Dataset integrity hash
    # --------------------------------------------------------

    file_hash = hashlib.sha256()

    with open(
        ground_truth_path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            file_hash.update(
                chunk
            )

    GROUND_TRUTH_REGISTRY.append({

        "dataset_id":
            dataset_id,

        "path":
            str(ground_truth_path),

        "rows":
            int(
                ground_truth_df.shape[0]
            ),

        "columns":
            int(
                ground_truth_df.shape[1]
            ),

        "sha256":
            file_hash.hexdigest(),

        "status":
            "VERIFIED"
    })

    del ground_truth_df

print("=" * 90)
print("GROUND-TRUTH DATASETS VERIFIED")
print("=" * 90)

GROUND_TRUTH_REGISTRY_DF = pd.DataFrame(
    GROUND_TRUTH_REGISTRY
)

display(
    GROUND_TRUTH_REGISTRY_DF
)

GROUND-TRUTH DATASETS VERIFIED


,dataset_id,path,rows,columns,sha256,status
0,adult_income,/content/drive/MyDrive/AIR_LLM_Research/data/s...,19522,14,12c5362bd5b7902fd24f5acf2736cac060e71e01d1988b...,VERIFIED
1,bank_marketing,/content/drive/MyDrive/AIR_LLM_Research/data/s...,27126,16,e9e9c2dd4e66498ad903060731e52e04c6144ac46397b0...,VERIFIED
2,diabetes_130us,/content/drive/MyDrive/AIR_LLM_Research/data/s...,61059,47,4f016a449b5e121e149bcf315891fdfee4a47d14384039...,VERIFIED


In [19]:
# ============================================================
# 03.14B SAVE GROUND-TRUTH REGISTRY
# ============================================================

GROUND_TRUTH_REGISTRY_PATH = (
    SCENARIO_ROOT /
    "ground_truth_registry.csv"
)

GROUND_TRUTH_REGISTRY_DF.to_csv(
    GROUND_TRUTH_REGISTRY_PATH,
    index=False
)

print("=" * 90)
print("GROUND-TRUTH REGISTRY SAVED")
print("=" * 90)

print(
    f"Registry:\n"
    f"{GROUND_TRUTH_REGISTRY_PATH}"
)

print()
print(
    "Ground-truth training data remains "
    "unchanged and stored in data/splits/."
)

GROUND-TRUTH REGISTRY SAVED
Registry:
/content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/ground_truth_registry.csv

Ground-truth training data remains unchanged and stored in data/splits/.


In [20]:
# ============================================================
# 03.15 MISSINGNESS VALIDATION
# ============================================================

import gc
import numpy as np
import pandas as pd

VALIDATION_RECORDS = []


# ------------------------------------------------------------
# VALIDATION FUNCTION
# ------------------------------------------------------------

def validate_saved_scenario(
    original_df,
    masked_df,
    mask_df,
    expected_rate,
    target_column=None
):
    """
    Validate one saved missingness scenario.

    Checks:
    1. Dataset dimensions are unchanged.
    2. Only intended cells became missing.
    3. Existing observed values are unchanged.
    4. Target is never masked.
    5. Actual missingness rate is recorded.
    """

    # --------------------------------------------------------
    # Shape validation
    # --------------------------------------------------------

    shape_preserved = (
        original_df.shape ==
        masked_df.shape
    )

    if not shape_preserved:

        raise AssertionError(
            "Dataset shape changed during masking."
        )

    # --------------------------------------------------------
    # Mask dimensions
    # --------------------------------------------------------

    mask_columns = list(
        mask_df.columns
    )

    if target_column in mask_columns:

        raise AssertionError(
            f"Target column '{target_column}' "
            f"must not appear in the missingness mask."
        )

    # --------------------------------------------------------
    # Actual missingness rate
    # --------------------------------------------------------

    actual_rate = float(
        mask_df.to_numpy(
            dtype=np.float64
        ).mean()
    )

    # --------------------------------------------------------
    # Verify every masked column
    # --------------------------------------------------------

    values_preserved = True
    unexpected_missingness = False
    changed_observed_values = 0

    for column in mask_columns:

        original_series = (
            original_df[column]
        )

        masked_series = (
            masked_df[column]
        )

        column_mask = (
            mask_df[column].to_numpy(
                dtype=bool
            )
        )

        # ----------------------------------------------------
        # Masked cells must actually be missing
        # ----------------------------------------------------

        invalid_masked_cells = (
            column_mask
            &
            ~masked_series.isna().to_numpy()
        )

        if invalid_masked_cells.any():

            unexpected_missingness = True

        # ----------------------------------------------------
        # Unmasked cells must remain unchanged
        # ----------------------------------------------------

        observed_mask = (
            ~column_mask
        )

        original_observed = (
            original_series
            .iloc[
                np.flatnonzero(
                    observed_mask
                )
            ]
            .reset_index(drop=True)
        )

        masked_observed = (
            masked_series
            .iloc[
                np.flatnonzero(
                    observed_mask
                )
            ]
            .reset_index(drop=True)
        )

        comparison = (
            original_observed
            .eq(masked_observed)
        )

        # NaN == NaN is False, so explicitly
        # handle pre-existing missing values.

        both_missing = (
            original_observed.isna()
            &
            masked_observed.isna()
        )

        valid_comparison = (
            comparison
            |
            both_missing
        )

        differences = (
            ~valid_comparison
        ).sum()

        changed_observed_values += int(
            differences
        )

    # --------------------------------------------------------
    # Target preservation
    # --------------------------------------------------------

    target_preserved = True

    if target_column is not None:

        target_preserved = (
            original_df[
                target_column
            ].equals(
                masked_df[
                    target_column
                ]
            )
        )

    # --------------------------------------------------------
    # Overall validation
    # --------------------------------------------------------

    values_preserved = (
        changed_observed_values == 0
    )

    scenario_valid = (
        shape_preserved
        and
        values_preserved
        and
        target_preserved
        and
        not unexpected_missingness
    )

    return {

        "expected_rate":
            float(expected_rate),

        "actual_rate":
            actual_rate,

        "rate_difference":
            float(
                abs(
                    actual_rate -
                    expected_rate
                )
            ),

        "shape_preserved":
            bool(shape_preserved),

        "observed_values_preserved":
            bool(values_preserved),

        "changed_observed_values":
            int(
                changed_observed_values
            ),

        "target_preserved":
            bool(target_preserved),

        "scenario_valid":
            bool(scenario_valid)
    }


# ------------------------------------------------------------
# LOAD CELL-WISE SCENARIO REGISTRY
# ------------------------------------------------------------

CELL_REGISTRY_PATH = (
    SCENARIO_ROOT /
    "cell_wise_scenario_registry.csv"
)

if not CELL_REGISTRY_PATH.exists():

    raise FileNotFoundError(
        f"Cell-wise scenario registry not found:\n"
        f"{CELL_REGISTRY_PATH}\n\n"
        "Run Notebook 03.11 first."
    )


CELL_WISE_REGISTRY_DF = pd.read_csv(
    CELL_REGISTRY_PATH
)

print("=" * 90)
print("03.15 MISSINGNESS VALIDATION")
print("=" * 90)

print(
    f"Scenarios discovered: "
    f"{len(CELL_WISE_REGISTRY_DF):,}"
)

03.15 MISSINGNESS VALIDATION
Scenarios discovered: 135


In [21]:
# ============================================================
# 03.15B VALIDATE SAVED CELL-WISE SCENARIOS
# ============================================================

for dataset_id in DATASET_IDS:

    print()
    print(
        "-" * 90
    )
    print(
        f"Validating: {dataset_id}"
    )
    print(
        "-" * 90
    )

    # --------------------------------------------------------
    # Load original training data ONCE
    # --------------------------------------------------------

    ground_truth_path = (
        PROJECT_ROOT /
        "data" /
        "splits" /
        dataset_id /
        "X_train.csv"
    )

    if not ground_truth_path.exists():

        raise FileNotFoundError(
            f"Ground-truth training file not found:\n"
            f"{ground_truth_path}"
        )

    original_df = pd.read_csv(
        ground_truth_path,
        low_memory=False
    )

    dataset_registry = (
        CELL_WISE_REGISTRY_DF[
            CELL_WISE_REGISTRY_DF[
                "dataset_id"
            ] == dataset_id
        ]
    )

    for _, record in dataset_registry.iterrows():

        data_path = Path(
            record["data_path"]
        )

        mask_path = Path(
            record["mask_path"]
        )

        if not data_path.exists():

            print(
                f"WARNING: Missing data file:\n"
                f"{data_path}"
            )

            continue

        if not mask_path.exists():

            print(
                f"WARNING: Missing mask file:\n"
                f"{mask_path}"
            )

            continue

        # ----------------------------------------------------
        # Load ONE scenario
        # ----------------------------------------------------

        masked_df = pd.read_csv(
            data_path,
            low_memory=False
        )

        mask_df = pd.read_csv(
            mask_path,
            low_memory=False
        )

        # ----------------------------------------------------
        # Validate
        # ----------------------------------------------------

        validation = validate_saved_scenario(

            original_df=original_df,

            masked_df=masked_df,

            mask_df=mask_df,

            expected_rate=float(
                record[
                    "missing_rate"
                ]
            ),

            target_column=record.get(
                "target_column",
                None
            )
        )

        VALIDATION_RECORDS.append({

            "dataset_id":
                dataset_id,

            "scenario":
                record[
                    "scenario"
                ],

            "mechanism":
                record[
                    "mechanism"
                ],

            "missing_rate":
                float(
                    record[
                        "missing_rate"
                    ]
                ),

            "actual_rate":
                validation[
                    "actual_rate"
                ],

            "rate_difference":
                validation[
                    "rate_difference"
                ],

            "seed":
                int(
                    record[
                        "seed"
                    ]
                ),

            "shape_preserved":
                validation[
                    "shape_preserved"
                ],

            "observed_values_preserved":
                validation[
                    "observed_values_preserved"
                ],

            "changed_observed_values":
                validation[
                    "changed_observed_values"
                ],

            "target_preserved":
                validation[
                    "target_preserved"
                ],

            "scenario_valid":
                validation[
                    "scenario_valid"
                ],

            "data_path":
                str(
                    data_path
                ),

            "mask_path":
                str(
                    mask_path
                )
        })

        # ----------------------------------------------------
        # Release scenario immediately
        # ----------------------------------------------------

        del masked_df
        del mask_df

        gc.collect()

    # --------------------------------------------------------
    # Release original dataset
    # --------------------------------------------------------

    del original_df

    gc.collect()


# ------------------------------------------------------------
# CREATE VALIDATION TABLE
# ------------------------------------------------------------

VALIDATION_DF = pd.DataFrame(
    VALIDATION_RECORDS
)

print()
print("=" * 90)
print("MISSINGNESS VALIDATION COMPLETED")
print("=" * 90)

print(
    f"Scenarios validated: "
    f"{len(VALIDATION_DF):,}"
)

if not VALIDATION_DF.empty:

    print(
        f"Valid scenarios: "
        f"{VALIDATION_DF['scenario_valid'].sum():,}"
    )

    print(
        f"Invalid scenarios: "
        f"{(~VALIDATION_DF['scenario_valid']).sum():,}"
    )

display(
    VALIDATION_DF.head(20)
)


------------------------------------------------------------------------------------------
Validating: adult_income
------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------
Validating: bank_marketing
------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------
Validating: diabetes_130us
------------------------------------------------------------------------------------------

MISSINGNESS VALIDATION COMPLETED
Scenarios validated: 135
Valid scenarios: 135
Invalid scenarios: 0


,dataset_id,scenario,mechanism,missing_rate,actual_rate,rate_difference,seed,shape_preserved,observed_values_preserved,changed_observed_values,target_preserved,scenario_valid,data_path,mask_path
0,adult_income,mcar_10pct_seed_42,MCAR,0.1,0.100001,7.317751e-07,42,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
1,adult_income,mcar_10pct_seed_123,MCAR,0.1,0.100001,7.317751e-07,123,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
2,adult_income,mcar_10pct_seed_2024,MCAR,0.1,0.100001,7.317751e-07,2024,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
3,adult_income,mcar_20pct_seed_42,MCAR,0.2,0.200001,1.463550e-06,42,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
4,adult_income,mcar_20pct_seed_123,MCAR,0.2,0.200001,1.463550e-06,123,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
5,adult_income,mcar_20pct_seed_2024,MCAR,0.2,0.200001,1.463550e-06,2024,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
6,adult_income,mcar_30pct_seed_42,MCAR,0.3,0.299999,1.463550e-06,42,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
7,adult_income,mcar_30pct_seed_123,MCAR,0.3,0.299999,1.463550e-06,123,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
8,adult_income,mcar_30pct_seed_2024,MCAR,0.3,0.299999,1.463550e-06,2024,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...
9,adult_income,mcar_40pct_seed_42,MCAR,0.4,0.399999,7.317751e-07,42,True,True,0,True,True,/content/drive/MyDrive/AIR_LLM_Research/experi...,/content/drive/MyDrive/AIR_LLM_Research/experi...


In [22]:
# ============================================================
# 03.15C VALIDATION SUMMARY
# ============================================================

if VALIDATION_DF.empty:

    raise RuntimeError(
        "No scenarios were validated."
    )

# ------------------------------------------------------------
# Hard validation checks
# ------------------------------------------------------------

if not (
    VALIDATION_DF[
        "shape_preserved"
    ].all()
):

    raise AssertionError(
        "Some scenarios changed dataset shape."
    )

if not (
    VALIDATION_DF[
        "observed_values_preserved"
    ].all()
):

    raise AssertionError(
        "Some observed values were modified."
    )

if not (
    VALIDATION_DF[
        "target_preserved"
    ].all()
):

    raise AssertionError(
        "Target preservation failed."
    )

if not (
    VALIDATION_DF[
        "scenario_valid"
    ].all()
):

    invalid = VALIDATION_DF[
        ~VALIDATION_DF[
            "scenario_valid"
        ]
    ]

    raise AssertionError(
        f"{len(invalid)} invalid "
        "missingness scenarios detected."
    )

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

VALIDATION_SUMMARY = (
    VALIDATION_DF
    .groupby(
        [
            "dataset_id",
            "mechanism",
            "missing_rate"
        ],
        as_index=False
    )
    .agg(

        scenarios=(
            "scenario",
            "count"
        ),

        mean_actual_rate=(
            "actual_rate",
            "mean"
        ),

        mean_rate_difference=(
            "rate_difference",
            "mean"
        ),

        all_valid=(
            "scenario_valid",
            "all"
        )
    )
)

display(
    VALIDATION_SUMMARY
)

,dataset_id,mechanism,missing_rate,scenarios,mean_actual_rate,mean_rate_difference,all_valid
0,adult_income,MAR,0.1,3,0.101322,1.321586e-03,True
1,adult_income,MAR,0.2,3,0.202802,2.801723e-03,True
2,adult_income,MAR,0.3,3,0.304039,4.039155e-03,True
3,adult_income,MAR,0.4,3,0.405334,5.333909e-03,True
4,adult_income,MAR,0.5,3,0.507354,7.354340e-03,True
5,adult_income,MCAR,0.1,3,0.100001,7.317751e-07,True
6,adult_income,MCAR,0.2,3,0.200001,1.463550e-06,True
7,adult_income,MCAR,0.3,3,0.299999,1.463550e-06,True
8,adult_income,MCAR,0.4,3,0.399999,7.317751e-07,True
9,adult_income,MCAR,0.5,3,0.500000,0.000000e+00,True


In [23]:
# ============================================================
# 03.15D SAVE MISSINGNESS VALIDATION
# ============================================================

VALIDATION_PATH = (
    SCENARIO_ROOT /
    "missingness_validation.csv"
)

SUMMARY_PATH = (
    SCENARIO_ROOT /
    "missingness_validation_summary.csv"
)

VALIDATION_DF.to_csv(
    VALIDATION_PATH,
    index=False
)

VALIDATION_SUMMARY.to_csv(
    SUMMARY_PATH,
    index=False
)

print("=" * 90)
print("VALIDATION RESULTS SAVED")
print("=" * 90)

print(
    f"Detailed validation:\n"
    f"{VALIDATION_PATH}"
)

print(
    f"\nValidation summary:\n"
    f"{SUMMARY_PATH}"
)

print()
print(
    "All saved cell-wise scenarios passed "
    "ground-truth and missingness validation."
)

VALIDATION RESULTS SAVED
Detailed validation:
/content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/missingness_validation.csv

Validation summary:
/content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/missingness_validation_summary.csv

All saved cell-wise scenarios passed ground-truth and missingness validation.


In [24]:
# ============================================================
# 03.16 SAVE EXPERIMENTAL SCENARIOS
# ============================================================

import gc
import json
import pandas as pd
from pathlib import Path

SCENARIO_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 90)
print("03.16 FINALIZING EXPERIMENTAL SCENARIOS")
print("=" * 90)


# ------------------------------------------------------------
# 1. VERIFY CELL-WISE SCENARIOS
# ------------------------------------------------------------

CELL_REGISTRY_PATH = (
    SCENARIO_ROOT /
    "cell_wise_scenario_registry.csv"
)

if not CELL_REGISTRY_PATH.exists():

    raise FileNotFoundError(
        f"Cell-wise scenario registry not found:\n"
        f"{CELL_REGISTRY_PATH}\n\n"
        "Run Notebook 03.11 first."
    )

CELL_WISE_REGISTRY_DF = pd.read_csv(
    CELL_REGISTRY_PATH
)

print(
    f"Cell-wise scenarios: "
    f"{len(CELL_WISE_REGISTRY_DF):,}"
)


# ------------------------------------------------------------
# 2. VERIFY FEATURE-WISE SCENARIOS
# ------------------------------------------------------------

FEATURE_REGISTRY_PATH = (
    SCENARIO_ROOT /
    "feature_wise" /
    "feature_wise_scenario_registry.csv"
)

if FEATURE_REGISTRY_PATH.exists():

    FEATURE_WISE_REGISTRY_DF = (
        pd.read_csv(
            FEATURE_REGISTRY_PATH
        )
    )

    print(
        f"Feature-wise scenarios: "
        f"{len(FEATURE_WISE_REGISTRY_DF):,}"
    )

else:

    FEATURE_WISE_REGISTRY_DF = (
        pd.DataFrame()
    )

    print(
        "Feature-wise registry not found."
    )


# ------------------------------------------------------------
# 3. COMBINE EXPERIMENTAL REGISTRIES
# ------------------------------------------------------------

registry_frames = []

if not CELL_WISE_REGISTRY_DF.empty:

    registry_frames.append(
        CELL_WISE_REGISTRY_DF
    )

if not FEATURE_WISE_REGISTRY_DF.empty:

    registry_frames.append(
        FEATURE_WISE_REGISTRY_DF
    )

if not registry_frames:

    raise RuntimeError(
        "No experimental scenario registries "
        "were found."
    )

EXPERIMENTAL_SCENARIO_REGISTRY = (
    pd.concat(
        registry_frames,
        ignore_index=True,
        sort=False
    )
)


# ------------------------------------------------------------
# 4. STANDARDIZE REGISTRY COLUMNS
# ------------------------------------------------------------

if "actual_missing_rate" in (
    EXPERIMENTAL_SCENARIO_REGISTRY.columns
):

    EXPERIMENTAL_SCENARIO_REGISTRY[
        "missing_rate_actual"
    ] = (
        EXPERIMENTAL_SCENARIO_REGISTRY[
            "actual_missing_rate"
        ]
    )

elif "missing_rate_actual" not in (
    EXPERIMENTAL_SCENARIO_REGISTRY.columns
):

    EXPERIMENTAL_SCENARIO_REGISTRY[
        "missing_rate_actual"
    ] = np.nan


# ------------------------------------------------------------
# 5. VERIFY ALL REFERENCED FILES
# ------------------------------------------------------------

print()
print("=" * 90)
print("VERIFYING SCENARIO FILES")
print("=" * 90)

missing_data_files = []
missing_mask_files = []

for _, row in (
    EXPERIMENTAL_SCENARIO_REGISTRY.iterrows()
):

    data_path = Path(
        row["data_path"]
    )

    mask_path = Path(
        row["mask_path"]
    )

    if not data_path.exists():

        missing_data_files.append(
            str(data_path)
        )

    if not mask_path.exists():

        missing_mask_files.append(
            str(mask_path)
        )


if missing_data_files:

    print(
        f"Missing data files: "
        f"{len(missing_data_files):,}"
    )

    for path in missing_data_files[:10]:

        print(
            f"  {path}"
        )

    raise FileNotFoundError(
        "One or more scenario data files "
        "are missing."
    )


if missing_mask_files:

    print(
        f"Missing mask files: "
        f"{len(missing_mask_files):,}"
    )

    for path in missing_mask_files[:10]:

        print(
            f"  {path}"
        )

    raise FileNotFoundError(
        "One or more scenario mask files "
        "are missing."
    )


print(
    "All referenced scenario files verified."
)


# ------------------------------------------------------------
# 6. LOAD EXISTING VALIDATION RESULTS
# ------------------------------------------------------------

VALIDATION_PATH = (
    SCENARIO_ROOT /
    "missingness_validation.csv"
)

if VALIDATION_PATH.exists():

    VALIDATION_DF = pd.read_csv(
        VALIDATION_PATH
    )

    print(
        f"Validation records: "
        f"{len(VALIDATION_DF):,}"
    )

else:

    print(
        "WARNING: Missingness validation "
        "file was not found."
    )

    VALIDATION_DF = pd.DataFrame()


# ------------------------------------------------------------
# 7. FINAL SCENARIO REGISTRY
# ------------------------------------------------------------

REGISTRY_PATH = (
    SCENARIO_ROOT /
    "scenario_registry.csv"
)

EXPERIMENTAL_SCENARIO_REGISTRY.to_csv(
    REGISTRY_PATH,
    index=False
)


# ------------------------------------------------------------
# 8. SAVE VALIDATION COPY
# ------------------------------------------------------------

if not VALIDATION_DF.empty:

    VALIDATION_DF.to_csv(
        VALIDATION_PATH,
        index=False
    )


# ------------------------------------------------------------
# 9. CREATE EXPERIMENT SUMMARY
# ------------------------------------------------------------

SUMMARY_RECORDS = []

for dataset_id in DATASET_IDS:

    dataset_registry = (
        EXPERIMENTAL_SCENARIO_REGISTRY[
            EXPERIMENTAL_SCENARIO_REGISTRY[
                "dataset_id"
            ] == dataset_id
        ]
    )

    if dataset_registry.empty:

        continue

    for scenario_type in [
        "cell_wise",
        "feature_wise"
    ]:

        if scenario_type == "cell_wise":

            subset = dataset_registry[
                ~dataset_registry[
                    "scenario"
                ].str.startswith(
                    "feature_",
                    na=False
                )
            ]

        else:

            subset = dataset_registry[
                dataset_registry[
                    "scenario"
                ].str.startswith(
                    "feature_",
                    na=False
                )
            ]

        if subset.empty:

            continue

        SUMMARY_RECORDS.append({

            "dataset_id":
                dataset_id,

            "scenario_type":
                scenario_type,

            "scenarios":
                int(
                    len(subset)
                ),

            "mechanisms":
                int(
                    subset[
                        "mechanism"
                    ]
                    .nunique()
                )
                if "mechanism"
                in subset.columns
                else 0,

            "missingness_levels":
                int(
                    subset[
                        "missing_rate"
                    ]
                    .nunique()
                ),

            "seeds":
                int(
                    subset[
                        "seed"
                    ]
                    .nunique()
                )
                if "seed"
                in subset.columns
                else 0
        })


SCENARIO_SUMMARY_DF = pd.DataFrame(
    SUMMARY_RECORDS
)

SUMMARY_PATH = (
    SCENARIO_ROOT /
    "experimental_scenario_summary.csv"
)

SCENARIO_SUMMARY_DF.to_csv(
    SUMMARY_PATH,
    index=False
)


# ------------------------------------------------------------
# 10. SAVE EXPERIMENT METADATA
# ------------------------------------------------------------

EXPERIMENT_METADATA = {

    "datasets":
        list(DATASET_IDS),

    "mechanisms":
        list(MECHANISMS),

    "missingness_levels":
        list(
            MISSINGNESS_LEVELS
        ),

    "random_seeds":
        list(
            RANDOM_SEEDS
        ),

    "cell_wise_scenarios":
        int(
            len(
                CELL_WISE_REGISTRY_DF
            )
        ),

    "feature_wise_scenarios":
        int(
            len(
                FEATURE_WISE_REGISTRY_DF
            )
        ),

    "total_scenarios":
        int(
            len(
                EXPERIMENTAL_SCENARIO_REGISTRY
            )
        ),

    "validation_records":
        int(
            len(
                VALIDATION_DF
            )
        ),

    "status":
        "FINALIZED"
}

METADATA_PATH = (
    SCENARIO_ROOT /
    "experimental_metadata.json"
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        EXPERIMENT_METADATA,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 11. FINAL OUTPUT
# ------------------------------------------------------------

print()
print("=" * 90)
print("NOTEBOOK 03 — EXPERIMENTAL SCENARIOS FINALIZED")
print("=" * 90)

print(
    f"Cell-wise scenarios   : "
    f"{len(CELL_WISE_REGISTRY_DF):,}"
)

print(
    f"Feature-wise scenarios : "
    f"{len(FEATURE_WISE_REGISTRY_DF):,}"
)

print(
    f"Total scenarios        : "
    f"{len(EXPERIMENTAL_SCENARIO_REGISTRY):,}"
)

print()
print(
    f"Scenario registry:\n"
    f"{REGISTRY_PATH}"
)

print(
    f"\nValidation:\n"
    f"{VALIDATION_PATH}"
)

print(
    f"\nSummary:\n"
    f"{SUMMARY_PATH}"
)

print(
    f"\nMetadata:\n"
    f"{METADATA_PATH}"
)

# ------------------------------------------------------------
# MEMORY CLEANUP
# ------------------------------------------------------------

gc.collect()

print()
print(
    "Notebook 03 experimental artifacts "
    "successfully finalized."
)

03.16 FINALIZING EXPERIMENTAL SCENARIOS
Cell-wise scenarios: 135
Feature-wise registry not found.

VERIFYING SCENARIO FILES
All referenced scenario files verified.
Validation records: 135

NOTEBOOK 03 — EXPERIMENTAL SCENARIOS FINALIZED
Cell-wise scenarios   : 135
Feature-wise scenarios : 0
Total scenarios        : 135

Scenario registry:
/content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/scenario_registry.csv

Validation:
/content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/missingness_validation.csv

Summary:
/content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/experimental_scenario_summary.csv

Metadata:
/content/drive/MyDrive/AIR_LLM_Research/experiments/missingness/experimental_metadata.json

Notebook 03 experimental artifacts successfully finalized.


In [25]:
# ============================================================
# 03.17 FINAL EXPERIMENTAL SCENARIO VERIFICATION
# ============================================================

print("=" * 90)
print("FINAL NOTEBOOK 03 VERIFICATION")
print("=" * 90)

for dataset_id in DATASET_IDS:

    dataset_dir = (
        SCENARIO_ROOT /
        dataset_id
    )

    scenario_files = [
        p for p in dataset_dir.glob("*.csv")
        if not p.name.endswith("_mask.csv")
    ]

    mask_files = list(
        dataset_dir.glob("*_mask.csv")
    )

    print()
    print(dataset_id)

    print(
        f"  Scenario files : "
        f"{len(scenario_files):,}"
    )

    print(
        f"  Mask files     : "
        f"{len(mask_files):,}"
    )

    assert len(scenario_files) > 0
    assert len(mask_files) > 0

assert REGISTRY_PATH.exists()
assert VALIDATION_PATH.exists()

print()
print("=" * 90)
print("NOTEBOOK 03 COMPLETED SUCCESSFULLY")
print("=" * 90)

print(
    "Experimental scenarios are now available "
    "for Notebook 04."
)

FINAL NOTEBOOK 03 VERIFICATION

adult_income
  Scenario files : 45
  Mask files     : 45

bank_marketing
  Scenario files : 45
  Mask files     : 45

diabetes_130us
  Scenario files : 45
  Mask files     : 45

NOTEBOOK 03 COMPLETED SUCCESSFULLY
Experimental scenarios are now available for Notebook 04.
